In [1]:
import pandas as pd

# Mock financial data: Reference text contains transaction metadata
data = {
    "tx_id": [101, 102, 103],
    "amount": [50000, 20, 95000],
    "description": [
        "URGENT WIRE TRANSFER TO OFFSHORE SHELL CO LUXEMBOURG", 
        "Starbucks coffee purchase", 
        "STRUCTURED CASH DEPOSITS UNDER TEN THOUSAND LIMIT"
    ]
}
df = pd.DataFrame(data)


In [2]:
from transformers import pipeline

# Loading a lightweight financial/risk classification model
classifier = pipeline("text-classification", model="ProsusAI/finbert")


/home/rahul-changezi/mypython/tx_monitoring/testenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12865.18it/s]


In [3]:
def flag_transaction(row):
    # Rule 1: High amount threshold
    if row['amount'] > 10000:
        return "FLAGGED: HIGH VALUE"

    # Rule 2: AI Sentiment / Risk analysis on description
    result = classifier(row['description'])[0]
    if result['label'] == 'negative' and result['score'] > 0.7:
        return f"FLAGGED: RISK TEXT ({result['score']:.2f})"

    return "CLEAN"

df['status'] = df.apply(flag_transaction, axis=1)
print(df)



   tx_id  amount                                        description  \
0    101   50000  URGENT WIRE TRANSFER TO OFFSHORE SHELL CO LUXE...   
1    102      20                          Starbucks coffee purchase   
2    103   95000  STRUCTURED CASH DEPOSITS UNDER TEN THOUSAND LIMIT   

                status  
0  FLAGGED: HIGH VALUE  
1                CLEAN  
2  FLAGGED: HIGH VALUE  


In [4]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="ProsusAI/finbert"
)

text = """
The bank reported strong earnings and improved liquidity.
"""

result = classifier(text)

print(result)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7028.62it/s]


[{'label': 'positive', 'score': 0.9588487148284912}]
